# GT-MedRAG — Notebook 04: Grounded-GRPO Respondent Agent

GRPO-trains the **respondent agent** (paper Sec. 3.3) — but with a reward design that goes beyond the paper: instead of a binary QA-answer match (impossible for open-ended reports), the reward is **verifiable grounding**, computed by the repository's own verification code (`src/generation/postprocess.py`):

| Reward component | Weight | Means |
| --- | --- | --- |
| R_support | 0.4 | fraction of report sentences lexically supported by retrieved passages |
| R_citation | 0.25 | fraction of `[E#]` tags whose cited passage supports the sentence |
|
| R_structure | 0.15 | all required sections present (FINDINGS/DIFFERENTIAL/INVESTIGATIONS/REFERENCES) |
| R_conflict | 0.2 | mentions CONFLICTS appropriately when evidence contradicts (paper gap addressed) |

Training pairs: `data/processed/sft_pairs.jsonl` prompts (clean) + `data/processed/noise_variants.jsonl` prompts (adversarial noise curriculum: drop/swap/contradict at 20%/40%) so the respondent learns robustness to corrupted retrieval — the paper's admitted failure mode.

In [ ]:
!nvidia-smi -L
!pip -q install transformers datasets accelerate trl peft sentence-transformers rank_bm25 pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
REPO = '/content/medical-cdss'
DRIVE = '/content/drive/MyDrive/medical-cdss'
BASE = f'{DRIVE}/gpt2-clinical-base'
OUT = f'{DRIVE}/gpt2-clinical-rag'
if not os.path.isdir(REPO):
    !git clone https://github.com/sharon504/medical-cdss.git $REPO
%cd $REPO

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

clean = [json.loads(l) for l in open(f'{REPO}/data/processed/sft_pairs.jsonl')]
noisy = [json.loads(l) for l in open(f'{REPO}/data/processed/noise_variants.jsonl')]
print(f'{len(clean)} clean + {len(noisy)} noisy training prompts')

# group dataset: each item carries the prompt and its evidence ids so the
# reward function can ground its check against the right passages
rows = []
for rec in clean + noisy:
    rows.append({
        'prompt': rec['prompt'],
        'condition': rec['condition'],
        'corruption': rec.get('corruption', 'clean'),
        'rate': rec.get('rate', 0.0),
        'evidence_ids': rec.get('evidence_ids', []),
    })
train_ds = Dataset.from_list(rows)
train_ds = train_ds.shuffle(seed=19)
print(train_ds)

In [ ]:
import re, sys
sys.path.insert(0, f'{REPO}')
from src.retrieval.bm25_index import search_corpora, kb_paths_from_settings
import pickle

with open(kb_paths_from_settings()['index'], 'rb') as fh:
    bundle = pickle.load(fh)
CHUNKS = bundle['chunks']
BY_ID = {c['chunk_id']: c['text'] for c in CHUNKS}

STOP = {'the','a','an','of','and','or','to','in','is','are','with','for','on',
        'may','be','was','were','patients','patient','associated','common',
        'typically','usually','such','as'}
def covered(sentence: str, passage: str, min_hits: int = 4) -> bool:
    words = [w for w in re.sub(r'[^a-z0-9 ]', ' ', sentence.lower()).split()
             if w not in STOP]
    p = re.sub(r'[^a-z0-9 ]', ' ', passage.lower())
    return sum(w in p for w in words) >= min_hits

REQUIRED = ('FINDINGS', 'DIFFERENTIAL', 'INVESTIGATIONS', 'REFERENCES')

def grounded_reward(completions, evidence_ids=None, corruption='clean', **kw):
    """Grounding-as-Reward: paper has binary QA match; we verify claims."""
    rewards = []
    for comp, ev_ids in zip(completions, evidence_ids):
        passages = [BY_ID.get(cid, '') for cid in (ev_ids or [])]
        sents = [s for s in re.split(r'(?<=[.!?])\s+', comp) if len(s.split()) >= 6]
        if sents:
            supported = sum(any(covered(s, p) for p in passages) for s in sents)
            r_support = supported / len(sents)
        else:
            r_support = 0.0
        tags = re.findall(r'\[(E\d+)\]', comp)
        if tags:
            hits = 0
            for sent in [s for s in re.split(r'(?<=[.!?])\s+', comp) if s.strip()]:
                for tag in re.findall(r'\[(E\d+)\]', sent):
                    idx = int(tag[1:]) - 1
                    if idx < len(passages) and covered(sent, passages[idx]):
                        hits += 1
            r_citation = hits / len(tags)
        else:
            r_citation = 0.0
        upper = comp.upper()
        r_structure = sum(r in upper for r in REQUIRED) / len(REQUIRED)
        r_conflict = 1.0 if corruption != 'clean' and 'CONFLICT' in upper else \
                     (1.0 if corruption == 'clean' else 0.0)
        rewards.append(
            0.40*r_support + 0.25*r_citation + 0.15*r_structure + 0.20*r_conflict)
    return rewards

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
import glob, json, os, shutil

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token

SFT_DIR = f'{DRIVE}/gpt2-clinical-rag-sft'        # restored notebook-02 SFT model
GRPO_DIR = f'{DRIVE}/gpt2-clinical-rag-final'     # final artifact of this notebook

# one-time restore of the SFT stage: an earlier GRPO run saved over
# gpt2-clinical-rag, so rebuild the SFT init from notebook 02's checkpoint
# folder (trainer_state.json records which checkpoint had the best eval_loss).
if not os.path.isdir(SFT_DIR):
    states = glob.glob(f'{DRIVE}/ckpt-clinical-rag/checkpoint-*/trainer_state.json')
    assert states, ('no SFT checkpoints found - run notebook 02 once, or copy '
                    'gpt2-clinical-rag to gpt2-clinical-rag-sft manually')
    best_state = max(states, key=os.path.getmtime)
    best_ckpt = json.load(open(best_state)).get('best_model_checkpoint') \
        or best_state.replace('/trainer_state.json', '')
    if not os.path.isabs(best_ckpt):
        best_ckpt = os.path.join(DRIVE, best_ckpt)
    shutil.copytree(best_ckpt, SFT_DIR)
    tok.save_pretrained(SFT_DIR)
    print('restored SFT init ->', SFT_DIR, '(from', best_ckpt + ')')

model = AutoModelForCausalLM.from_pretrained(SFT_DIR)

# manual prompt truncation (this TRL version has no prompt-length kwarg):
# cap prompts at 736 tokens so prompt + completion (256) stays inside GPT-2's
# 1024-position limit.
MAX_PROMPT_TOKENS = 736
def _trunc_prompt(rec):
    ids = tok(rec['prompt'], add_special_tokens=False,
              truncation=True, max_length=MAX_PROMPT_TOKENS)['input_ids']
    return {'prompt': tok.decode(ids, skip_special_tokens=True)}
train_ds = train_ds.map(_trunc_prompt)

cfg = GRPOConfig(
    output_dir=f'{DRIVE}/ckpt-grpo-respondent',
    learning_rate=5e-6,                      # paper respondent LR
    per_device_train_batch_size=2,
    gradient_accumulation_steps=5,           # 2x5=10 = 2 prompts x G=5 completions
    num_train_epochs=1,
    num_generations=5,                       # paper's group size G=5
    max_completion_length=256,
    beta=0.01,                               # KL penalty (paper)
    temperature=0.7,
    fp16=False,                              # fp16 overflows on T4 GRPO (loss blew up 127 -> 6.6e7); fp32 is stable
    logging_steps=5,
    save_strategy='epoch',
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    args=cfg,
    train_dataset=train_ds,
    processing_class=tok,
    reward_funcs=[grounded_reward],
)
trainer.train()
trainer.save_model(GRPO_DIR)
tok.save_pretrained(GRPO_DIR)
print('Grounded-GRPO respondent saved ->', GRPO_DIR)


In [ ]:
# stability study: token-level output entropy per config
# run AFTER training completes (measure stability, paper Sec. 4.11 style)
import json, os, sys
sys.path.insert(0, f'{REPO}')
from src.generation.pipeline import ReportGenerator
from src.evaluation.robustness import token_entropy

SFT_DIR = globals().get('SFT_DIR', f'{DRIVE}/gpt2-clinical-rag-sft')
GRPO_DIR = globals().get('GRPO_DIR', f'{DRIVE}/gpt2-clinical-rag-final')

patient = json.load(open(f'{REPO}/configs/sample_patient.json'))
entropy_rows = {}
for label, ckpt, mode in [
    ('base_raw', BASE, 'raw'),
    ('base_triplets', BASE, 'triplets'),
    ('sft_triplets', SFT_DIR, 'triplets'),
    ('post_grpo_triplets', GRPO_DIR, 'triplets'),
]:
    if not os.path.isdir(ckpt):
        print(f'skipping {label}: {ckpt} missing')
        continue
    gen = ReportGenerator(model_path=ckpt, mode=mode)
    prompt, _, _ = gen.retrieve_endpoints(patient)
    entropy_rows[label] = token_entropy(gen, prompt, n_samples=16)

print(json.dumps(entropy_rows, indent=2))
if 'sft_triplets' in entropy_rows and 'post_grpo_triplets' in entropy_rows:
    expect_lower_after = entropy_rows['post_grpo_triplets'] < entropy_rows['sft_triplets']
    print('lower entropy after Grounded-GRPO (expected, paper-consistent):', expect_lower_after)
